### Load LEI Data and filter out unnecessary columns

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
from matplotlib import pyplot as plt

cols = ['LEI', 'Entity.LegalName','Entity.RegistrationAuthority.RegistrationAuthorityID', 'Entity.RegistrationAuthority.RegistrationAuthorityEntityID', 
        'Entity.RegistrationAuthority.OtherRegistrationAuthorityID','Entity.LegalJurisdiction','Entity.EntityStatus','Registration.RegistrationStatus', 'Registration.ManagingLOU']

lei_data = pd.read_csv('D:/SVL/20211202-0800-gleif-goldencopy-lei2-golden-copy.csv.zip',
    compression='zip',
    low_memory=False,
    dtype=str,  # put it as a string always
                       
    # the following will prevent pandas from converting words like 'NA' to NaN. We want to work with the LEI data as it is.
    na_values=[''],
    keep_default_na=False, usecols = cols, nrows= 500000) # we are taking 1000 rows, due to lack of memory

# lei_data.info()
lei_data
# lei_data[lei_data['Entity.LegalJurisdiction'] == 'AZ']

,LEI,Entity.LegalName,Entity.RegistrationAuthority.RegistrationAuthorityID,Entity.RegistrationAuthority.OtherRegistrationAuthorityID,Entity.RegistrationAuthority.RegistrationAuthorityEntityID,Entity.LegalJurisdiction,Entity.EntityStatus,Registration.RegistrationStatus,Registration.ManagingLOU
0,001GPB6A9XPE8XJICC14,FIDELITY ADVISOR SERIES I - Fidelity Advisor L...,RA000665,NaN,S000005113,US-MA,ACTIVE,ISSUED,EVK05KS7XY1DEII3R011
1,004L5FPTUREIWK9T2N63,"Hutchin Hill Capital, LP",RA000602,NaN,4386463,US-DE,ACTIVE,LAPSED,EVK05KS7XY1DEII3R011
2,00EHHQ2ZHDCFXJCPCL46,Vanguard Russell 1000 Growth Index Trust,RA999999,NaN,NaN,US-PA,ACTIVE,ISSUED,EVK05KS7XY1DEII3R011
3,00GBW0Z2GYIER7DHDS71,"ARISTEIA CAPITAL, L.L.C.",RA000602,NaN,2758229,US-DE,ACTIVE,ISSUED,EVK05KS7XY1DEII3R011
4,00KLB2PFTM3060S2N216,HARRIS ASSOCIATES INVESTMENT TRUST - Oakmark I...,RA000665,NaN,S000002762,US-MA,ACTIVE,ISSUED,EVK05KS7XY1DEII3R011
...,...,...,...,...,...,...,...,...,...
499995,391200DJEO9DD8V61G88,Wrangel Real Estate Consulting & Investment GmbH,RA000304,NaN,HRB 133553,DE,ACTIVE,ISSUED,39120001KULK7200U106
499996,391200DJETE3NVXT1T81,Provaluta Aktiengesellschaft,RA000296,NaN,HRB 8794,DE,ACTIVE,LAPSED,39120001KULK7200U106
499997,391200DJEWSG4KL9TH87,Stiftung des Hohenlohekreises,RA999999,NaN,NaN,DE,ACTIVE,ISSUED,39120001KULK7200U106
499998,391200DJFGWJNXNCLX52,Stiftung Dresdner Kreuzchor - Förderstiftung -,RA999999,NaN,NaN,DE,ACTIVE,ISSUED,39120001KULK7200U106


In [119]:
# data_merged[data_merged.LEI=='213800FPUQ7KORJYOZ90']
# ra_list[ra_list['Registration Authority Code']=='RA000192']
lei_data[lei_data['Entity.RegistrationAuthority.RegistrationAuthorityID'] =='RA000192'].head(20)

# lei_data[lei_data.LEI == '213800FPUQ7KORJYOZ90']
# lei_data[lei_data.LEI == '222100N2525B35AQVN31']
# lei_data.duplicated().sum()
lei_data.loc[lei_data.duplicated(), :] # shows the rows that identified as duplicates


# lei_data.duplicated(subset = 'Entity.RegistrationAuthority.RegistrationAuthorityID')

0

### Return the column labels

In [25]:
## Read only the columns we are interested in
# lei_data.columns # doesn't show all the columns
list(lei_data.columns) # shows all the columns as a list

['LEI',
 'Entity.LegalName',
 'Entity.RegistrationAuthority.RegistrationAuthorityID',
 'Entity.RegistrationAuthority.OtherRegistrationAuthorityID',
 'Entity.RegistrationAuthority.RegistrationAuthorityEntityID',
 'Entity.LegalJurisdiction',
 'Entity.EntityStatus',
 'Registration.RegistrationStatus',
 'Registration.ManagingLOU']

### Filter out columns. We only need RA related ones and ManagingLOU.

In [10]:
# read multiply columns at the same time
# lei_mod = lei_data[['LEI', 'Entity.RegistrationAuthority.RegistrationAuthorityID', 'Entity.RegistrationAuthority.RegistrationAuthorityEntityID'
         # 'Entity.RegistrationAuthority.OtherRegistrationAuthorityID','Registration.ManagingLOU']] # we only left 6 columns
# lei_mod # modified lei data



# lei_mod = lei_data.drop('Entity.LegalName','', axis=1, inplace = True) # axis=1 => horizontal, axis=0 => vertical

### Read RA list

In [2]:
ra_list = pd.read_excel('D:/SVL/2021-06-16_ra_list_v1.6.xlsx')
# Read each column / a specific column
# print(ra_list["Registration Authority Code"][0:6]) # specify top 6 RA codes
# print(ra_list[["Registration Authority Code", "Jurisdiction (country or region)"]]) 
ra_list
#ra_list.loc[ra_list['Country'] == 'Azerbaijan'] # access only the rows that have a specific data. Focus on a specific rows.


,Registration Authority Code,Country,Country Code,Jurisdiction (country or region),International name of Register,Local name of Register,International name of organisation responsible for the Register,Local name of organisation responsible for the Register,Website,Comments
0,RA777777,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,to be used for 'Public Legal Documents' for Ge...
1,RA888888,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,temporary code; new registration authority cod...
2,RA999999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no registration authority available for this l...
3,RA000001,Afghanistan,AF,Afghanistan,Afghanistan Central Business Registry (Ministr...,NaN,Afghanistan Central Business Registry (Ministr...,NaN,http://www.acbr.gov.af/,NaN
4,RA000759,Afghanistan,AF,Afghanistan,Register of Licensed Financial Institutions,NaN,Da Afghanistan Bank,NaN,https://dab.gov.af/index.php/Licensed-Finacial...,to be used for banks and some other licensed f...
...,...,...,...,...,...,...,...,...,...,...
769,RA000192,Wallis and Futuna,WF,Wallis and Futuna,Register of Commerce and Companies,Registre du Commerce et des Sociétés,Infogreffe,Infogreffe,www.infogreffe.com,NaN
770,RA000650,Wallis and Futuna,WF,Wallis and Futuna,NaN,NaN,NaN,Service de la Réglementation et des Élections ...,http://www.wallis-et-futuna.pref.gouv.fr/Vos-d...,NaN
771,RA000651,Yemen,YE,Yemen,NaN,NaN,Ministry of Industry and Trade,وزارة الصناعة والتجارة,http://www.moit.gov.ye/moit/,NaN
772,RA000652,Zambia,ZM,Zambia,Companies Register,Companies Register,Patents and Companies Registration Agency (PACRA),Patents and Companies Registration Agency (PACRA),www.pacra.org.zm,NaN


In [133]:
ra_list[['Registration Authority Code', 'International name of Register']].value_counts() # which RA's do cover (have) more than one Jurisdiction
# this are sorted by value descending, we have the top registers and the 5th one already appears only once. All below that are all appearing once. So we only have four RA that have more than 2 Jurisdictions.

Registration Authority Code  International name of Register                                             
RA000192                     Register of Commerce and Companies                                             12
RA000189                     Register of Companies (Sirene)                                                 12
RA000745                     Registry of federally insured credit unions                                     4
RA000548                     UID-Register                                                                    2
RA000001                     Afghanistan Central Business Registry (Ministry of Commerce and Industries)     1
                                                                                                            ..
RA000242                     Commercial Register                                                             1
RA000241                     Commercial Register                                                             1
RA00024

In [123]:
ra_list.duplicated() # shows duplicates in the RA list after removing the duplicates
# ra_list.duplicated().sum()

0      False
1      False
2      False
3      False
4      False
       ...  
769    False
770    False
771    False
772    False
773    False
Length: 774, dtype: bool

In [4]:
ra_list[ra_list['Registration Authority Code']=='RA000189']

,Registration Authority Code,Country,Country Code,Jurisdiction (country or region),International name of Register,Local name of Register,International name of organisation responsible for the Register,Local name of organisation responsible for the Register,Website,Comments
215,RA000189,France,FR,France,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
218,RA000189,French Guiana,GF,French Guiana,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
220,RA000189,French Polynesia,PF,French Polynesia,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
409,RA000189,Guadeloupe,GP,Guadeloupe,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
498,RA000189,Martinique,MQ,Martinique,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
503,RA000189,Mayotte,YT,Mayotte,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
523,RA000189,New Caledonia,NC,New Caledonia,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
571,RA000189,Réunion,RE,Réunion,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
573,RA000189,Saint Barthélemy,BL,Saint Barthélemy,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
578,RA000189,Saint Martin,MF,Saint Martin,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN


### Droping duplicates in the entire Dataset

In [91]:
ra_list.drop_duplicates(keep = 'first', inplace = True) # remove duplicates in RA list 
ra_list

,Registration Authority Code,Country,Country Code,Jurisdiction (country or region),International name of Register,Local name of Register,International name of organisation responsible for the Register,Local name of organisation responsible for the Register,Website,Comments
0,RA777777,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,to be used for 'Public Legal Documents' for Ge...
1,RA888888,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,temporary code; new registration authority cod...
2,RA999999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no registration authority available for this l...
3,RA000001,Afghanistan,AF,Afghanistan,Afghanistan Central Business Registry (Ministr...,NaN,Afghanistan Central Business Registry (Ministr...,NaN,http://www.acbr.gov.af/,NaN
4,RA000759,Afghanistan,AF,Afghanistan,Register of Licensed Financial Institutions,NaN,Da Afghanistan Bank,NaN,https://dab.gov.af/index.php/Licensed-Finacial...,to be used for banks and some other licensed f...
...,...,...,...,...,...,...,...,...,...,...
767,RA000649,Vietnam,VN,Vietnam,Business Registration,NaN,Agency of Business Registration (Ministry of P...,NaN,https://dangkykinhdoanh.gov.vn/en/Pages/defaul...,NaN
770,RA000650,Wallis and Futuna,WF,Wallis and Futuna,NaN,NaN,NaN,Service de la Réglementation et des Élections ...,http://www.wallis-et-futuna.pref.gouv.fr/Vos-d...,NaN
771,RA000651,Yemen,YE,Yemen,NaN,NaN,Ministry of Industry and Trade,وزارة الصناعة والتجارة,http://www.moit.gov.ye/moit/,NaN
772,RA000652,Zambia,ZM,Zambia,Companies Register,Companies Register,Patents and Companies Registration Agency (PACRA),Patents and Companies Registration Agency (PACRA),www.pacra.org.zm,NaN


### Finding Duplicate Values in a Specific Column

In [83]:
ra_list.duplicated(subset = '')

0      False
1       True
2       True
3      False
4       True
       ...  
767    False
770    False
771    False
772    False
773    False
Length: 748, dtype: bool

In [109]:
ra_countries = ra_list.groupby('Registration Authority Code')['Country Code'].apply(list) # all the countries are in the list with the particular RA codes
ra_countries

# ra_countries = ra_list.groupby('Registration Authority Code')['Country Code'].apply(list).loc['RA000189']
# ra_countries

Registration Authority Code
RA000001     [AF]
RA000002     [AL]
RA000003     [DZ]
RA000004     [AD]
RA000005     [AO]
            ...  
RA000791     [GB]
RA000792     [US]
RA777777    [nan]
RA888888    [nan]
RA999999    [nan]
Name: Country Code, Length: 748, dtype: object

In [88]:
ra_countries.get('RA000665') 

['US']

In [106]:
lei_data.assign(
    jurisdiction_coveredby_ra=lei_data.fillna('--').apply(lambda row: row['Entity.LegalJurisdiction'][:2] in ra_countries.get(row['Entity.RegistrationAuthority.RegistrationAuthorityID']), axis = 1)
).jurisdiction_coveredby_ra.value_counts()

# lei_data.assign(
#     jurisdiction_coveredby_ra=lei_data.fillna('--').apply(lambda row: row['Entity.LegalJurisdiction'][:3] in ra_countries.get(row['Entity.RegistrationAuthority.RegistrationAuthorityID']), axis = 1)
# )

,LEI,Entity.LegalName,Entity.RegistrationAuthority.RegistrationAuthorityID,Entity.RegistrationAuthority.OtherRegistrationAuthorityID,Entity.RegistrationAuthority.RegistrationAuthorityEntityID,Entity.LegalJurisdiction,Entity.EntityStatus,Registration.RegistrationStatus,Registration.ManagingLOU,jurisdiction_coveredby_ra
0,001GPB6A9XPE8XJICC14,FIDELITY ADVISOR SERIES I - Fidelity Advisor L...,RA000665,NaN,S000005113,US-MA,ACTIVE,ISSUED,EVK05KS7XY1DEII3R011,False
1,004L5FPTUREIWK9T2N63,"Hutchin Hill Capital, LP",RA000602,NaN,4386463,US-DE,ACTIVE,LAPSED,EVK05KS7XY1DEII3R011,False
2,00EHHQ2ZHDCFXJCPCL46,Vanguard Russell 1000 Growth Index Trust,RA999999,NaN,NaN,US-PA,ACTIVE,ISSUED,EVK05KS7XY1DEII3R011,False
3,00GBW0Z2GYIER7DHDS71,"ARISTEIA CAPITAL, L.L.C.",RA000602,NaN,2758229,US-DE,ACTIVE,ISSUED,EVK05KS7XY1DEII3R011,False
4,00KLB2PFTM3060S2N216,HARRIS ASSOCIATES INVESTMENT TRUST - Oakmark I...,RA000665,NaN,S000002762,US-MA,ACTIVE,ISSUED,EVK05KS7XY1DEII3R011,False
...,...,...,...,...,...,...,...,...,...,...
499995,391200DJEO9DD8V61G88,Wrangel Real Estate Consulting & Investment GmbH,RA000304,NaN,HRB 133553,DE,ACTIVE,ISSUED,39120001KULK7200U106,True
499996,391200DJETE3NVXT1T81,Provaluta Aktiengesellschaft,RA000296,NaN,HRB 8794,DE,ACTIVE,LAPSED,39120001KULK7200U106,True
499997,391200DJEWSG4KL9TH87,Stiftung des Hohenlohekreises,RA999999,NaN,NaN,DE,ACTIVE,ISSUED,39120001KULK7200U106,False
499998,391200DJFGWJNXNCLX52,Stiftung Dresdner Kreuzchor - Förderstiftung -,RA999999,NaN,NaN,DE,ACTIVE,ISSUED,39120001KULK7200U106,False


In [95]:
ra_list[ra_list['Registration Authority Code']=='RA000192'] # according to RA code

,Registration Authority Code,Country,Country Code,Jurisdiction (country or region),International name of Register,Local name of Register,International name of organisation responsible for the Register,Local name of organisation responsible for the Register,Website,Comments
217,RA000192,France,FR,France,Register of Commerce and Companies,Registre du Commerce et des Sociétés,Infogreffe,Infogreffe,www.infogreffe.com,NaN


### Drop duplicates

In [121]:
ra_list.drop_duplicates(subset = 'Registration Authority Code', keep = 'first') # remove duplicates in a specific column
ra_list

,Registration Authority Code,Country,Country Code,Jurisdiction (country or region),International name of Register,Local name of Register,International name of organisation responsible for the Register,Local name of organisation responsible for the Register,Website,Comments
0,RA777777,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,to be used for 'Public Legal Documents' for Ge...
1,RA888888,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,temporary code; new registration authority cod...
2,RA999999,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,no registration authority available for this l...
3,RA000001,Afghanistan,AF,Afghanistan,Afghanistan Central Business Registry (Ministr...,NaN,Afghanistan Central Business Registry (Ministr...,NaN,http://www.acbr.gov.af/,NaN
4,RA000759,Afghanistan,AF,Afghanistan,Register of Licensed Financial Institutions,NaN,Da Afghanistan Bank,NaN,https://dab.gov.af/index.php/Licensed-Finacial...,to be used for banks and some other licensed f...
...,...,...,...,...,...,...,...,...,...,...
769,RA000192,Wallis and Futuna,WF,Wallis and Futuna,Register of Commerce and Companies,Registre du Commerce et des Sociétés,Infogreffe,Infogreffe,www.infogreffe.com,NaN
770,RA000650,Wallis and Futuna,WF,Wallis and Futuna,NaN,NaN,NaN,Service de la Réglementation et des Élections ...,http://www.wallis-et-futuna.pref.gouv.fr/Vos-d...,NaN
771,RA000651,Yemen,YE,Yemen,NaN,NaN,Ministry of Industry and Trade,وزارة الصناعة والتجارة,http://www.moit.gov.ye/moit/,NaN
772,RA000652,Zambia,ZM,Zambia,Companies Register,Companies Register,Patents and Companies Registration Agency (PACRA),Patents and Companies Registration Agency (PACRA),www.pacra.org.zm,NaN


In [67]:
ra_list[ra_list['Registration Authority Code']=='RA000192']

,Registration Authority Code,Country,Country Code,Jurisdiction (country or region),International name of Register,Local name of Register,International name of organisation responsible for the Register,Local name of organisation responsible for the Register,Website,Comments
217,RA000192,France,FR,France,Register of Commerce and Companies,Registre du Commerce et des Sociétés,Infogreffe,Infogreffe,www.infogreffe.com,NaN


### Merge LEI Data and RA list

In [68]:
# data_merged = pd.merge(lei_data, ra_list, left_on = 'Entity.RegistrationAuthority.RegistrationAuthorityID', right_on='Registration Authority Code', how='left')
data_merged = pd.merge(lei_data, ra_list, left_on = 'Entity.RegistrationAuthority.RegistrationAuthorityID', right_on='Registration Authority Code', how='outer')
data_merged

# lei_data.columns
# ra_list.columns

#lei_data['temp'] = 1
#ra_list['temp'] = 1
#data_merged = pd.merge(lei_data, ra_list, on='temp') # on = 'common_column'
#data_merged = data_merged.drop('temp', axis=1, inplace = False) # axis 1 - horizontal # inplace=False didn't work?
#data_merged
# lei_data # we need to have 15 columns in merged lei dataset and 5 in original lei data. Extra 2 columns tmp & add in merged data set and extra temp column in original; why?



,LEI,Entity.LegalName,Entity.RegistrationAuthority.RegistrationAuthorityID,Entity.RegistrationAuthority.OtherRegistrationAuthorityID,Entity.RegistrationAuthority.RegistrationAuthorityEntityID,Entity.LegalJurisdiction,Entity.EntityStatus,Registration.RegistrationStatus,Registration.ManagingLOU,Registration Authority Code,Country,Country Code,Jurisdiction (country or region),International name of Register,Local name of Register,International name of organisation responsible for the Register,Local name of organisation responsible for the Register,Website,Comments
0,001GPB6A9XPE8XJICC14,FIDELITY ADVISOR SERIES I - Fidelity Advisor L...,RA000665,NaN,S000005113,US-MA,ACTIVE,ISSUED,EVK05KS7XY1DEII3R011,RA000665,United States of America,US,United States of America,EDGAR,EDGAR,Securities and Exchange Commission,Securities and Exchange Commission,https://www.sec.gov/edgar/searchedgar/webusers...,NaN
1,00KLB2PFTM3060S2N216,HARRIS ASSOCIATES INVESTMENT TRUST - Oakmark I...,RA000665,NaN,S000002762,US-MA,ACTIVE,ISSUED,EVK05KS7XY1DEII3R011,RA000665,United States of America,US,United States of America,EDGAR,EDGAR,Securities and Exchange Commission,Securities and Exchange Commission,https://www.sec.gov/edgar/searchedgar/webusers...,NaN
2,00X8DSV26QKJPKUT5B34,"T. Rowe Price Summit Municipal Funds, Inc. - T...",RA000665,NaN,S000002157,US-MD,ACTIVE,ISSUED,EVK05KS7XY1DEII3R011,RA000665,United States of America,US,United States of America,EDGAR,EDGAR,Securities and Exchange Commission,Securities and Exchange Commission,https://www.sec.gov/edgar/searchedgar/webusers...,NaN
3,01PZ5A1XH5933KUUB187,TIFF INVESTMENT PROGRAM - TIFF Multi-Asset Fund,RA000665,NaN,S000006387,US-DE,ACTIVE,ISSUED,EVK05KS7XY1DEII3R011,RA000665,United States of America,US,United States of America,EDGAR,EDGAR,Securities and Exchange Commission,Securities and Exchange Commission,https://www.sec.gov/edgar/searchedgar/webusers...,NaN
4,024DELVYES1ONBTJWD69,PUTNAM FUNDS TRUST - Putnam Focused Equity Fund,RA000665,NaN,S000024246,US-MA,ACTIVE,ISSUED,EVK05KS7XY1DEII3R011,RA000665,United States of America,US,United States of America,EDGAR,EDGAR,Securities and Exchange Commission,Securities and Exchange Commission,https://www.sec.gov/edgar/searchedgar/webusers...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
500116,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RA000756,Turkey,TR,Turkey,Public Disclosure Platform,Kamuyu Aydınlatma Platformu (KAP),Central Securities Depository of Turkey,Merkezi Kayıt Kuruluşu Anonim Şirketi (MKK),https://www.kap.org.tr/en/,to be used for funds in Turkey
500117,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RA000564,Tuvalu,TV,Tuvalu,Companies and Business Registration Register,NaN,Registrar of Companies,NaN,www.tuvaluislands.com/gov_addresses.htm,NaN
500118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RA000788,United Kingdom,GB,England,NaN,NaN,The Church of England,The Church of England,https://www.achurchnearyou.com/,to be used for churches / parishes of the Chur...
500119,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RA000790,United Kingdom,GB,United Kingdom,Register Of Statutory Auditors,Register Of Statutory Auditors,Register Of Statutory Auditors,Register Of Statutory Auditors,www.auditregister.org.uk,to be used for chartered accountant firms


In [84]:
data_merged.duplicated().sum() # number of duplicate records in the entire dataset

0

In [86]:
data_merged.duplicated(subset='LEI').sum() # number of duplicate values in a specific column.

120

In [87]:
data_merged.duplicated(subset='Registration Authority Code').sum()

499373

In [97]:
data_merged[data_merged['Registration Authority Code']=='RA000189']

,LEI,Entity.LegalName,Entity.RegistrationAuthority.RegistrationAuthorityID,Entity.RegistrationAuthority.OtherRegistrationAuthorityID,Entity.RegistrationAuthority.RegistrationAuthorityEntityID,Entity.LegalJurisdiction,Entity.EntityStatus,Registration.RegistrationStatus,Registration.ManagingLOU,Registration Authority Code,Country,Country Code,Jurisdiction (country or region),International name of Register,Local name of Register,International name of organisation responsible for the Register,Local name of organisation responsible for the Register,Website,Comments
295554,0XKTQGPTA3P542CII630,IBM FRANCE FINANCEMENT,RA000189,NaN,316037753,FR,ACTIVE,ISSUED,969500Q2MA9VBQ8BG884,RA000189,France,FR,France,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
295555,16TFY6OJ8XMF5ZY4AL63,GE Healthcare SAS,RA000189,NaN,303 215 123,FR,ACTIVE,LAPSED,EVK05KS7XY1DEII3R011,RA000189,France,FR,France,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
295556,1S619D6B3ZQIH6MS6B47,MILLEIS VIE,RA000189,NaN,384532172,FR,ACTIVE,ISSUED,969500Q2MA9VBQ8BG884,RA000189,France,FR,France,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
295557,1VUV7VQFKUOQSJ21A208,CREDIT AGRICOLE CORPORATE AND INVESTMENT BANK,RA000189,NaN,304187701,FR,ACTIVE,ISSUED,969500Q2MA9VBQ8BG884,RA000189,France,FR,France,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
295558,21380016W7GAG26FIJ74,SOCIETE FRANCAISE ET SUISSE,RA000189,NaN,326 028 347,FR,ACTIVE,ISSUED,213800WAVVOPS85N2205,RA000189,France,FR,France,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295910,3912000DJGJNMASJOV44,CAVOK CONSULTING,RA000189,NaN,437 739 709,FR,ACTIVE,ISSUED,39120001KULK7200U106,RA000189,France,FR,France,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
295911,3912002I6PRZUJ1XN897,FERME EOLIENNE DE LION EN BEAUCE,RA000189,NaN,833638844,FR,ACTIVE,ISSUED,39120001KULK7200U106,RA000189,France,FR,France,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
295912,3912005XBCU1FCO16A73,CENTRALE SOLAIRE DE DECIZE,RA000189,NaN,838 516 490,FR,ACTIVE,ISSUED,39120001KULK7200U106,RA000189,France,FR,France,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN
295913,391200A9BVKV6WV4AW87,Sci le ver luisant,RA000189,NaN,534 071 279,FR,ACTIVE,LAPSED,39120001KULK7200U106,RA000189,France,FR,France,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN


In [100]:
data_merged[data_merged.LEI=='213800FPUQ7KORJYOZ90'] # according to LEI 

,LEI,Entity.LegalName,Entity.RegistrationAuthority.RegistrationAuthorityID,Entity.RegistrationAuthority.OtherRegistrationAuthorityID,Entity.RegistrationAuthority.RegistrationAuthorityEntityID,Entity.LegalJurisdiction,Entity.EntityStatus,Registration.RegistrationStatus,Registration.ManagingLOU,Registration Authority Code,Country,Country Code,Jurisdiction (country or region),International name of Register,Local name of Register,International name of organisation responsible for the Register,Local name of organisation responsible for the Register,Website,Comments
443069,213800FPUQ7KORJYOZ90,EURL L'ECUREUIL,RA000192,NaN,842 002 990,FR,ACTIVE,ISSUED,213800WAVVOPS85N2205,RA000192,France,FR,France,Register of Commerce and Companies,Registre du Commerce et des Sociétés,Infogreffe,Infogreffe,www.infogreffe.com,NaN


In [101]:
data_merged[data_merged.LEI=='222100N2525B35AQVN31']

,LEI,Entity.LegalName,Entity.RegistrationAuthority.RegistrationAuthorityID,Entity.RegistrationAuthority.OtherRegistrationAuthorityID,Entity.RegistrationAuthority.RegistrationAuthorityEntityID,Entity.LegalJurisdiction,Entity.EntityStatus,Registration.RegistrationStatus,Registration.ManagingLOU,Registration Authority Code,Country,Country Code,Jurisdiction (country or region),International name of Register,Local name of Register,International name of organisation responsible for the Register,Local name of organisation responsible for the Register,Website,Comments
295864,222100N2525B35AQVN31,P.B.A.,RA000189,NaN,435369343,FR,ACTIVE,LAPSED,222100T6ICDIY8V4VX70,RA000189,France,FR,France,Register of Companies (Sirene),Sirene,National Institute of Statistics and Economic ...,Institut National de la Statistique et des Étu...,http://sirene.fr/sirene/public/accueil?sirene_...,NaN


### How many LEI per RA code and Jurisdiction

In [128]:
lei_ra=data_merged.groupby(['LEI','Registration Authority Code', 'Entity.LegalJurisdiction']).size().sort_values(ascending=False).rename('count').reset_index() 
lei_ra

# z = np.abs(stats.zscore(lei_ra))
#print(z)
# threshold = 3
# print(np.where(z > 3))

# lei_ra.hist(column='count')


# plt.hist(lei_ra., rwidth=0.8, bins = 30)
# plt.xlabel('RA code')
# plt.ylablel('counts')
# plt.show()

,LEI,Registration Authority Code,Entity.LegalJurisdiction,count
0,001GPB6A9XPE8XJICC14,RA000665,US-MA,1
1,254900VPKG31T93DN644,RA000629,US-OH,1
2,254900VPJOIE6I3C5151,RA000188,FI,1
3,254900VPJNCDR4TDUX69,RA000170,DK,1
4,254900VPJHFIBVMCO943,RA000629,US-OH,1
...,...,...,...,...
499989,213800TMEAJGGMIPAO45,RA000063,VG,1
499990,213800TMEAGF6ENKLK29,RA000587,GB,1
499991,213800TME2T7JWBU1F08,RA999999,HK,1
499992,213800TMDZACDPE4EP82,RA888888,GB,1


In [98]:
lei_ra[lei_ra['Registration Authority Code']=='RA000189']

,LEI,Registration Authority Code,Entity.LegalJurisdiction,count
4383,254900UMH2GAILRD5862,RA000189,FR,1
8328,254900XXAK1IQLF5W069,RA000189,FR,1
8734,254900XVLKHVHUD8RQ06,RA000189,FR,1
15505,254900X48HAOWT15VW13,RA000189,FR,1
18598,254900RSETZ8O7BNG212,RA000189,FR,1
...,...,...,...,...
497422,213800T6D8RTYCV9PM48,RA000189,FR,1
497427,213800T6CUFOAPGGF997,RA000189,FR,1
497892,213800T9DQ3RQPI28Q37,RA000189,FR,1
499236,213800TKXH6DZGUZPS47,RA000189,FR,1


In [99]:
lei_ra[lei_ra['Registration Authority Code']=='RA000192']

,LEI,Registration Authority Code,Entity.LegalJurisdiction,count
322,254900VMMIPSD3PN8F29,RA000192,FR,1
977,254900UB6OYILOA78Z09,RA000192,FR,1
1336,254900VDDF027XNWNL13,RA000192,FR,1
3950,254900UO9NTI6FVATO96,RA000192,FR,1
3980,254900UNE3S0E935ON47,RA000192,FR,1
...,...,...,...,...
498810,213800TS8CTOWSDO8U32,RA000192,FR,1
499223,213800TKOS7J5UA3ZI07,RA000192,FR,1
499238,213800TKWR72TT9CVI18,RA000192,FR,1
499669,213800TOFF59YBRW4L19,RA000192,FR,1


In [102]:
lei_ra[lei_ra['Registration Authority Code']=='RA000745']

,LEI,Registration Authority Code,Entity.LegalJurisdiction,count
294,254900VNHREN5BIMTO39,RA000745,US-VT,1
707,254900VVCX7PR9KDCK83,RA000745,US-ME,1
734,254900VRRL70B52CAL31,RA000745,US-NY,1
997,254900VFVL52CIXJ8D81,RA000745,US,1
1073,254900VFRQR5ZLXRQ298,RA000745,US-NY,1
...,...,...,...,...
436307,25490040IBIKTOL1DJ85,RA000745,US-PA,1
436569,2549003ODK1CMS9CMW53,RA000745,US,1
436699,2549003PJRW71IW7BH70,RA000745,US-MI,1
437093,2549003SGT19SUFFA317,RA000745,US-IL,1


In [72]:
jurisdiction=data_merged.groupby(['Country Code', 'Entity.LegalJurisdiction','Jurisdiction (country or region)', 'Country']).size().sort_values(ascending=False).rename('counts').reset_index()
jurisdiction
# jurisdiction.sort_values(by = ['Jurisdiction (country or region)'], ascending=False)

# jurisdiction.counts.describe()
# jurisdiction.hist(column='count')
# data_merged.groupby(['LEI','Jurisdiction (country or region)']).size().sort_values(ascending=False).rename('count').reset_index()

,Country Code,Entity.LegalJurisdiction,Jurisdiction (country or region),Country,counts
0,GB,GB,England and Wales,United Kingdom,56086
1,IN,IN,India,India,47282
2,CN,CN,China,China,29339
3,PL,PL,Poland,Poland,24914
4,CZ,CZ,Czech Republic,Czech Republic,18236
...,...,...,...,...,...
629,CO,CO,Neiva,Colombia,1
630,CO,CO,Monteria,Colombia,1
631,US,US-KY,Illinois,United States of America,1
632,SS,SS,South Sudan,South Sudan,1


### Which are the top RA codes

In [73]:
data_merged['Registration Authority Code'].value_counts(sort=True, ascending=False) # returns counts of RA codes

# most frequently occurring RA codes are RA999999 and RA000585


RA999999    99723
RA000585    47141
RA000394    30841
RA000092    29339
RA000484    18881
            ...  
RA000759        1
RA000016        1
RA000762        1
RA000707        1
RA000650        1
Name: Registration Authority Code, Length: 748, dtype: int64

### Which ManagingLOU is responsible for which RA

In [74]:
man_ra=data_merged.groupby(['LEI','Entity.LegalName','Registration.ManagingLOU','Registration Authority Code', 'Country', 'International name of Register']).size().rename('Count').reset_index()
man_ra.sort_values(by = ['Registration.ManagingLOU'], ascending=False)
# man_ra

# lou=man_ra.sort_values(by='Registration.ManagingLOU', ascending=False) 
# lou

# man_ra=data_merged.groupby(['Registration.ManagingLOU','Registration Authority Code', 'Country', 'International name of Register']).size()
# man_ra.sort_values(by = ['Registration.ManagingLOU'], ascending=False)
# man_ra



,LEI,Entity.LegalName,Registration.ManagingLOU,Registration Authority Code,Country,International name of Register,Count
0,001GPB6A9XPE8XJICC14,FIDELITY ADVISOR SERIES I - Fidelity Advisor L...,EVK05KS7XY1DEII3R011,RA000665,United States of America,EDGAR,1
107583,2221005KDR564YSEBJ20,INVESTMENT SOLUTIONS BY EPSILON - GLOBAL FLEXI...,EVK05KS7XY1DEII3R011,RA000433,Luxembourg,CSSF - Supervised Entities,1
108807,222100AK59BH7BBCNI33,Broad Street Loan Partners III Onshore S.à r.l.,EVK05KS7XY1DEII3R011,RA000432,Luxembourg,Trade and Company Register,1
108792,222100AH3I3WR79YPF10,PensPlan SICAV Lux-PensPlan Invest Euro Bond F...,EVK05KS7XY1DEII3R011,RA000433,Luxembourg,CSSF - Supervised Entities,1
108780,222100ADZQOI2AF5CH32,GOLDMAN SACHS FUNDS - GOLDMAN SACHS GLOBAL INF...,EVK05KS7XY1DEII3R011,RA000433,Luxembourg,CSSF - Supervised Entities,1
...,...,...,...,...,...,...,...
253,0292003713E6O0AD4G79,MAINSTREET BANK CAPITAL LIMITED,029200067A7K6CH0H586,RA000469,Nigeria,Company Registry,1
254,0292003728E9K1CI4J66,CITYCODE TRUST AND INVESTMENT LIMITED,029200067A7K6CH0H586,RA000469,Nigeria,Company Registry,1
255,0292003737D4P1AH3E44,EFG HERMES NIGERIA LIMITED,029200067A7K6CH0H586,RA000469,Nigeria,Company Registry,1
256,029200374B0P4B01A716,ROSTRUM INVESTMENT AND SECURITIES LIMITED,029200067A7K6CH0H586,RA000469,Nigeria,Company Registry,1


#### Do the RA list country codes match the Jurisdiction codes?

In [32]:
# codes=data_merged.groupby(['Country Code', 'Jurisdiction (country or region)']).size().sort_values(ascending=False).rename('counts').reset_index()
# codes

# codes.counts.describe()
# codes.hist(column='counts')

,Country Code,Jurisdiction (country or region),counts
0,GB,England and Wales,54346
1,LU,Luxembourg,13522
2,US,Delaware,10363
3,DK,Denmark,9759
4,VG,British Virgin Islands,9671
...,...,...,...
402,DE,Erfurt,1
403,DE,Greiz,1
404,DE,Göttingen,1
405,DE,Gütersloh,1


### Results as a single excel file

In [75]:
# data_merged.to_excel('D:/SVL/merged.xlsx')

results = pd.ExcelWriter('D:/SVL/AllDataNew.xlsx')

ra_list.to_excel(results, sheet_name = 'RA list')
data_merged.to_excel(results, sheet_name = 'RA list and LEI data') 
lei_ra.to_excel(results, sheet_name = 'Number of LEI per RA')
jurisdiction.to_excel(results, sheet_name = 'Jurisdiction Data')
man_ra.to_excel(results, sheet_name = 'ManagingLOU with RA')


# codes.to_excel(results, sheet_name = 'Country and Jurisdiction codes')

results.close()

### Detect the Outliers 


In [1]:
# Z score and standart deviation 
# done separately under the name "Finding animalies"